In [27]:
from pathlib import Path
import re
import math 
import pandas as pd

# 1. PDF extraction 
def extract_text_from_pdf(pdf_path):
    from pypdf import PdfReader
    reader = PdfReader(pdf_path)
    pages_text= []
    for i,page in enumerate(reader.pages):
        text = page.extract_text() or ""
        text =text.replace("\u00ad","")
        pages_text.append(text)

    full_text = "\n\n".join(pages_text).strip()
    return {"pages_text": pages_text, "full_text": full_text, "num_pages": len(pages_text)}

In [28]:
# 2. sanity checks 
def extraction_report(text:str)->dict: 
    #ration of normal characters to total characters
    printable = sum(c.isprintable() for c in text)
    total = max(1,len(text))
    printable_ratio = printable/total
    #roug word count 
    words = re.findall(r'\b\w+\b', text)
    word_count = len(words)
    # detect low signals 

    return {
        "char_count": len(text),
        "word_count": word_count,
        "printable_ratio": printable_ratio,
        "has_many_empty": word_count < 50 or len(text)<300
    }

def preview(text:str, n_char: int=1200)->str:
    return text[:n_char].repace("\n","\\n")

In [29]:
def preview(text:str, n_char: int=1200)->str:
    return text[:n_char].replace("\n","\\n")

In [30]:
# 3. Preprocessing

def basic_tokenization(text:str):
    return re.findall(r'\b\w+\b', text.lower())

def get_stopwords():
    try:
        from nltk.corpus import stopwords
        return set(stopwords.words("english"))
    except Exception:
        from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
        return set(ENGLISH_STOP_WORDS)

def stem_tokens(tokens):
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
    return [stemmer.stem(t) for t in tokens]

def lemmatize_tokens(tokens):
    """
    WordNet lemmatizer (requires NLTK wordnet corpora).
    If not available, it will raise; you can use stemming instead.
    """
    from nltk.stem import WordNetLemmatizer
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(t) for t in tokens]

def preprocess(text: str, mode: str = "stem"):
    """
    mode: "stem" or "lemma"
    """
    tokens = basic_tokenization(text)
    stop = get_stopwords()
    tokens_ns = [t for t in tokens if t not in stop and len(t) > 2]

    if mode == "stem":
        tokens_proc = stem_tokens(tokens_ns)
    elif mode == "lemma":
        tokens_proc = lemmatize_tokens(tokens_ns)
    else:
        raise ValueError("mode must be 'stem' or 'lemma'")

    return {
        "tokens_raw": tokens,
        "tokens_no_stop": tokens_ns,
        "tokens_processed": tokens_proc,
        "text_processed": " ".join(tokens_proc),
    }




In [31]:
# 4) TF-IDF / BoW + n-gram analysis
def top_ngrams(texts, method="tfidf", ngram_range=(1,2), top_k=20):
    """
    texts: list[str] documents
    method: "tfidf" or "bow"
    Returns DataFrame with top features by mean weight.
    """
    from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
    import numpy as np

    if method == "tfidf":
        vec = TfidfVectorizer(lowercase=True, ngram_range=ngram_range)
    elif method == "bow":
        vec = CountVectorizer(lowercase=True, ngram_range=ngram_range)
    else:
        raise ValueError("method must be 'tfidf' or 'bow'")

    X = vec.fit_transform(texts)
    feats = vec.get_feature_names_out()
    scores = X.mean(axis=0).A1  # average weight/count across docs
    idx = np.argsort(scores)[::-1][:top_k]

    return pd.DataFrame({"ngram": feats[idx], "score": scores[idx]})



In [32]:
# 5. Chunking 

def chunk_by_word(text:str, words_per_chunk=800, overlap=80):
    words=text.split()
    chunks= []
    start =0 
    while start < len(words):
        end = min(start + words_per_chunk, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break 
        start =max(0, end-overlap)
    return chunks

In [33]:
PDF_PATH ="eswa.pdf"

#1 Extraction 
data = extract_text_from_pdf(PDF_PATH)

In [34]:
data
full_text = data["full_text"]

In [35]:
print("Pages:",data["num_pages"])
rep = extraction_report(full_text)
print("Extraction Report:", rep)

Pages: 1
Extraction Report: {'char_count': 7133, 'word_count': 1182, 'printable_ratio': 0.9830365904948829, 'has_many_empty': False}


In [36]:
print("\nPreview----(first ~1200 chars)---")
print(preview(full_text))


Preview----(first ~1200 chars)---
Expert Systems With Applications 248 (2024) 123375\n11D.P. Panagoulias et al.\nTable 5\nClusters of perceived usefulness — AI literacy.\nQuestion 𝜔1(𝜀𝜗𝜛𝜚 ) 𝜔2(𝜀𝜗𝜛𝜚 ) Q(Average)\nQ1 3.40 3.0 3.20\nQ2 3.33 3.25 3.29\nQ3 3.96 2.91 3.70\nQ4 4.11 3.25 3.435\nQ5 4.55 2.75 3.65\nTotalCount 27 12\nAverage 3.87 3.03\nTable 6\nPerceived usefulness and associated AI literacy, survey questions.\nQuestion Perceived usefulness\nQ1 Can you indicate your level of knowledge on diagnostic\nmedicine\nQ2 Computer vision is a field of artificial intelligence that trains\ncomputers to interpret and understand the visual world.\nWould you trust it as a feature in driving automation\nQ3 AI can add value by either automating, assisting or\naugmenting the work of clinicians and staff. Many repetitive\ntasks will become fully automated, and we can also use AI\nas a tool to help health professionals perform better at their\njobs and improve outcomes for patients. How much do you

In [37]:
# 2. check per page
page_word_counts = [len(basic_tokenization(p)) for p in data["pages_text"]]
pd.DataFrame({"page": list(range(1, data["num_pages"] + 1)),
    "word_count": page_word_counts})

,page,word_count
0,1,1182


In [38]:
pp = preprocess(full_text, mode="stem")
processed_text = pp["text_processed"]

In [39]:
processed_text

'expert system applic 248 2024 123375 11d panagoulia tabl cluster perceiv use literaci question 𝜀𝜗𝜛𝜚 𝜀𝜗𝜛𝜚 averag 435 totalcount averag tabl perceiv use associ literaci survey question question perceiv use indic level knowledg diagnost medicin comput vision field artifici intellig train comput interpret understand visual world would trust featur drive autom add valu either autom assist augment work clinician staff mani repetit task becom fulli autom also use tool help health profession perform better job improv outcom patient much agre statement use empow system feel know underli technolog better trust provid would particip develop medic applic reinforc approach employ literaci metric extern variabl tabl 6illustr select question esti mate perceiv use correspond literaci gaug particip understand diagnost medicin domain employ statist method data analysi enhanc diagnosi treatment decis particip grasp diagnost medicin suggest comprehens statist data analysi thu hint elev literaci level aim

In [42]:
print("Tiken Count:")
print("Raw:", len(pp["tokens_raw"]))
print("tokens_no_stop", len(pp["tokens_no_stop"]))
print("text_processed", len(pp["text_processed"]))

Tiken Count:
Raw: 1182
tokens_no_stop 605
text_processed 4164
